### Missing Indicator

A **missing indicator** is an additional binary feature that records whether the original value was missing.

For example, an `Age` value can be imputed with the median, while a new `Age_NA` column records:
- `1` — the original `Age` value was missing.
- `0` — the original `Age` value was present.

This lets a model use both the imputed numerical value and the information that the value was missing.

#### Why use a Missing Indicator?
Imputation replaces missing values, but it can hide the fact that those values were missing. Sometimes the pattern of missingness itself contains useful information.

A missing indicator preserves that information as a separate feature. It is commonly combined with numerical or categorical imputation rather than used as a replacement for imputation.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

In [2]:
df = pd.read_csv('Datasets/titanic.csv',usecols=['Age','Fare','Survived'])

In [3]:
df.head()

,Survived,Age,Fare
0,0,22.0,7.2500
1,1,38.0,71.2833
2,1,26.0,7.9250
3,1,35.0,53.1000
4,0,35.0,8.0500


In [4]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [5]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=2)

In [6]:
X_train.head()

,Age,Fare
30,40.0,27.7208
10,4.0,16.7000
873,47.0,9.0000
182,9.0,31.3875
876,20.0,9.8458


In [7]:
si = SimpleImputer()
X_train_trf = si.fit_transform(X_train)
X_test_trf = si.transform(X_test)

In [8]:
X_train_trf

array([[ 40.        ,  27.7208    ],
       [  4.        ,  16.7       ],
       [ 47.        ,   9.        ],
       ...,
       [ 71.        ,  49.5042    ],
       [ 29.78590426, 221.7792    ],
       [ 29.78590426,  25.925     ]])

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

clf = LogisticRegression()
clf.fit(X_train_trf,y_train)

y_pred = clf.predict(X_test_trf)

print("Accuaracy Score Before Missing Indicator : " ,accuracy_score(y_test,y_pred))

Accuaracy Score Before Missing Indicator :  0.6145251396648045


#### 1. Creating indicators with `MissingIndicator`

Scikit-learn's `MissingIndicator` creates Boolean indicator columns for features that contain missing values.

- `fit(X_train)` identifies which features have missing values in the training data.
- `features_` shows the indices of those features.
- `transform(X_train)` and `transform(X_test)` create the corresponding indicator columns.

By default, `features='missing-only'` creates indicators only for columns that had missing values during fitting. Therefore, fit it on the training data and use the same fitted indicator to transform the test data.

In this notebook, the indicator is added to the dataset as `Age_NA`, alongside the imputed `Age` value.

In [10]:
from sklearn.impute import MissingIndicator
mi = MissingIndicator()

mi.fit(X_train)

MissingIndicator()

In [11]:
mi.features_

array([0])

In [12]:
X_train_missing = mi.transform(X_train)

In [13]:
X_train_missing

array([[False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [ True],
       [False],
       [False],
       [False],
       [ True],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [ True],
       [False],
       [False],
       [ True],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [ True],
       [ True],
       [False],
       [False],
       [False],
       [False],
       [ True],
       [False],
       [False],
       [False],
       [False],
       [ True],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [

In [14]:
X_test_missing = mi.transform(X_test)

In [15]:
X_test_missing

array([[False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [ True],
       [ True],
       [False],
       [ True],
       [False],
       [False],
       [False],
       [False],
       [ True],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [ True],
       [False],
       [False],
       [False],
       [False],
       [ True],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [ True],
       [False],
       [ True],
       [False],
       [False],
       [ True],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [

In [16]:
X_train['Age_NA'] = X_train_missing

In [17]:
X_test.tail()

,Age,Fare
89,24.0,8.0500
80,22.0,9.0000
846,NaN,69.5500
870,26.0,7.8958
251,29.0,10.4625


In [18]:
X_test['Age_NA'] = X_test_missing

In [19]:
X_train.tail()

,Age,Fare,Age_NA
534,30.0,8.6625,False
584,NaN,8.7125,True
493,71.0,49.5042,False
527,NaN,221.7792,True
168,NaN,25.9250,True


In [20]:
si = SimpleImputer()
X_train_trf = si.fit_transform(X_train)
X_test_trf = si.transform(X_test)

In [21]:
X_train_trf

array([[ 40.        ,  27.7208    ,   0.        ],
       [  4.        ,  16.7       ,   0.        ],
       [ 47.        ,   9.        ,   0.        ],
       ...,
       [ 71.        ,  49.5042    ,   0.        ],
       [ 29.78590426, 221.7792    ,   1.        ],
       [ 29.78590426,  25.925     ,   1.        ]])

In [22]:
clf = LogisticRegression()

clf.fit(X_train_trf,y_train)
y_pred = clf.predict(X_test_trf)

print("Accuracy Score after the missing indicators : ",accuracy_score(y_test,y_pred))

Accuracy Score after the missing indicators :  0.6312849162011173


So here we can the socre improve from 61% to 63% .

### 2. Creating indicators with `SimpleImputer`

`SimpleImputer(add_indicator=True)` combines imputation and missing-indicator creation in one transformer.

For each feature that contains missing values when the imputer is fitted, it appends a corresponding indicator feature to the transformed output. The imputed values and indicators are returned together.

This is a convenient option when you want to add missingness information without creating a separate `MissingIndicator` object.

In [23]:
from sklearn.impute import SimpleImputer
si = SimpleImputer()

X_train_trf2 = si.fit_transform(X_train)
X_test_trf2 = si.transform(X_test)

In [25]:
si = SimpleImputer(add_indicator=True)

In [26]:
X_train = si.fit_transform(X_train)

In [27]:
X_test = si.transform(X_test)

In [28]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression()

clf.fit(X_train,y_train)

y_pred = clf.predict(X_test)

from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.6312849162011173

#### When should you use a Missing Indicator?

Use a missing indicator when:
- Missingness may carry information about the target.
- You want the model to distinguish an originally missing value from an observed value that happens to equal the imputation value.
- You are already imputing missing values and want to preserve the missingness pattern.

It may not help when missingness is unrelated to the target. Adding unnecessary indicators can increase feature count and may add noise.

**Important practices**

- Fit the imputer and indicator using training data only.
- Apply the fitted transformers to validation and test data.
- Compare model performance with and without indicators using cross-validation.
- Remember that indicators reveal that a value was missing; they do not recover the original value.